# Q01 · Xu hướng mức độ nghiêm trọng

**Phụ trách:** A · **Tiêu chí:** D2, D3, D4

Tỷ lệ vụ nghiêm trọng thay đổi thế nào theo tháng và năm khi giữ cố định cơ cấu loại đường?

## 1. Động cơ và phạm vi

Ghi người dùng kết quả, đơn vị quan sát và giới hạn phạm vi.

In [ ]:
from pathlib import Path

import pandas as pd

# Notebook có thể chạy từ thư mục gốc hoặc từ thư mục notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "environment.yml").is_file():
    raise FileNotFoundError("Mở Jupyter Notebook từ thư mục gốc của dự án.")

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

In [ ]:
data_path = processed_dir / "accidents_clean.csv"
required_columns = ['accident_id', 'accident_date', 'accident_year', 'road_type', 'is_severe']

# Đọc vài dòng để kiểm tra bảng bàn giao trước khi xử lý dữ liệu thật.
if data_path.is_file():
    preview = pd.read_csv(data_path, nrows=5, dtype={"accident_id": "string"})
    missing_columns = sorted(set(required_columns) - set(preview.columns))
    if missing_columns:
        raise ValueError(f"{data_path.name} thiếu cột {missing_columns}")
    print(f"Đã kiểm tra các cột cần dùng trong {data_path.name}.")
else:
    print(f"Chưa có {data_path.name}. Chờ bảng do notebook 01 tạo.")

In [ ]:
accident_df=pd.read_csv(data_path,usecols=required_columns, dtype={"accident_id": "string"}, parse_dates=["accident_date"])

## 2. Chọn mẫu và kiểm tra dữ liệu

Chốt mẫu số, khoảng thời gian có dữ liệu và nhóm loại đường. Ghi số dòng trước và sau mỗi bước lọc.

In [ ]:
# Viết phần chọn mẫu và kiểm tra ở đây.

## 3. Phương pháp

Mô tả cách tính tỷ lệ chung, trọng số loại đường và chuỗi đã chuẩn hóa.

In [ ]:
# Viết phép tính và kiểm tra kết quả trung gian ở đây.

Tỷ lệ chuẩn hóa năm 2005 và năm 2015 đều là con số ước lượng từ các vụ được ghi nhận. Nếu số vụ ngẫu nhiên khác đi một chút, các con số này cũng sẽ khác. Bootstrap đo xem chúng có thể dao động bao nhiêu. Từ đó ta biết mức thay đổi 2005→2015 là thật, hay chỉ là nhiễu do mẫu.

2. Ý tưởng

1. Coi dữ liệu đang có là "tổng thể".
2. Rút lại có hoàn lại một mẫu cùng kích thước từ dữ liệu đó.
3. Tính lại đúng thống kê cần đo trên mẫu mới. Với Q01, đó là tỷ lệ chuẩn hóa của từng năm và chênh lệch giữa 2015 và 2005.
4. Lặp lại B lần, ví dụ B = 2000, để có B giá trị.
5. Khoảng tin cậy 95% là khoảng từ phân vị 2,5% đến 97,5% của B giá trị đó (gọi là phương pháp percentile).

3. Rút lại cái gì trong Q01

Đơn vị quan sát là vụ tai nạn, nên ta rút lại các vụ. Nên rút trong từng ô (năm, loại đường), gọi là bootstrap phân tầng. Lý do:

- Trọng số w_k của loại đường được giữ cố định theo định nghĩa, nên số vụ mỗi ô không cần thay đổi.
- Phần thực sự không chắc chắn là p_{k,t}, tỷ lệ nghiêm trọng trong từng ô.

Mẹo để chạy nhanh: dữ liệu có 1,78 triệu dòng, nếu rút lại từng dòng 2000 lần thì rất chậm. Vì is_severe chỉ nhận 0 hoặc 1, nên rút lại n vụ có hoàn lại trong một ô cho ra số vụ nghiêm trọng có cùng phân phối với Binomial(n, p̂). Vì vậy chỉ cần sinh số ngẫu nhiên Binomial cho từng ô, thay vì làm việc trên từng dòng.

4. Khung code

Mình đã chạy thử khung dưới đây trên dữ liệu thật, mất khoảng 0,2 giây. Mình cố ý không đưa kết quả: bạn tự chạy, tự đọc và tự diễn giải. Như vậy đúng yêu cầu ở mục 6 của PDF, và đến phần Q&A bạn cũng giải thích được.

import numpy as np

# Số vụ (n) và số vụ nghiêm trọng (k) trong từng ô năm × loại đường
cell = accident_df.groupby(["accident_year", "road_type"])["is_severe"].agg(n="size", k="sum")
n = cell["n"].unstack()          # hàng: năm, cột: loại đường
k = cell["k"].unstack()

# Trọng số cố định: cơ cấu loại đường của cả giai đoạn
w = accident_df["road_type"].value_counts(normalize=True).reindex(n.columns)

rng = np.random.default_rng(42)  # cố định seed để tái lập
B = 2000
# Mỗi lần lặp: sinh lại số vụ nghiêm trọng của từng ô ~ Binomial(n, p̂)
p_boot = rng.binomial(n.values, (k / n).values, size=(B, *n.shape)) / n.values
std_boot = (p_boot * w.values).sum(axis=2)       # (B, số năm): tỷ lệ chuẩn hóa

ci_year = np.percentile(std_boot, [2.5, 97.5], axis=0)   # CI cho từng năm
diff = std_boot[:, -1] - std_boot[:, 0]                  # 2015 − 2005
ci_diff = np.percentile(diff, [2.5, 97.5])

Lưu ý: nếu có ô nào n = 0, phép chia k / n ra NaN và cả kết quả bị hỏng. Hãy kiểm tra trước khi chạy, nhất là khi gom theo tháng.

5. Đọc kết quả

- CI của chênh lệch không chứa 0 thì mức thay đổi 2005→2015 không phải nhiễu do mẫu. Báo kèm cả giá trị chênh lệch, ví dụ "+x điểm phần trăm, CI 95% [a; b]".
- Trên biểu đồ: vẽ dải CI quanh đường tỷ lệ chuẩn hóa bằng fill_between.
- Tính theo tháng: mỗi ô ít vụ hơn nhiều, nên dải CI sẽ rộng hơn. Đây là một quan sát đáng ghi, vì nó cho thấy dao động theo tháng có thể chỉ là nhiễu.

6. Giới hạn cần ghi trong notebook

- Bootstrap chỉ đo nhiễu do mẫu. Nó không bắt được thay đổi có hệ thống, ví dụ cảnh sát đổi cách phân loại "nghiêm trọng" theo thời gian. CI hẹp không có nghĩa là kết luận chắc chắn đúng.
- Giả định các vụ độc lập. Thực tế các vụ trong cùng khu vực hoặc cùng tháng có thể tương quan, nên CI có thể hơi hẹp hơn thật. Có thể nêu block bootstrap (rút lại theo cụm, ví dụ theo police_force hoặc theo tháng) như một hướng mở rộng.
- Trọng số w_k được coi là cố định. Thực ra w_k cũng ước lượng từ dữ liệu, nhưng với 1,78 triệu vụ thì sai số của nó không đáng kể.

Cách giải thích ngắn gọn khi bị hỏi trong phần Q&A: "Em rút lại có hoàn lại các vụ trong từng ô năm × loại đường 2000 lần. Mỗi lần em tính lại tỷ lệ chuẩn hóa, rồi lấy phân vị 2,5% và 97,5% làm khoảng tin cậy."

## 4. Kết quả

Dự kiến một hình xu hướng và một hình mức thay đổi theo loại đường. Ghi tiêu đề, trục, đơn vị và cỡ mẫu.

In [ ]:
# Thêm các bảng và biểu đồ sau khi hoàn tất phân tích.

## 5. Trả lời và giới hạn

Điền kết luận dựa trên kết quả thật. Ghi ảnh hưởng của dữ liệu thiếu, cách chọn mẫu và các giới hạn diễn giải.